# Voice Humanization Notebook

This notebook processes existing audio files in the `outputs/` directory and applies humanization techniques to make AI-generated voices more natural.

## Features:
- Natural pause insertion
- Subtle pitch variation
- Audio quality enhancement
- Batch processing support



In [ ]:
# Cell 1: Environment Setup and Path Detection
# This cell detects whether you're running in Google Colab or locally
# and sets up the appropriate paths.

import os
import sys

# Detect environment
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("=" * 80)
print("VOICE HUMANIZATION - Environment Setup")
print("=" * 80)

if IN_COLAB:
    print("Detected: Google Colab environment")
    # Mount Google Drive
    print("\nMounting Google Drive...")
    drive.mount("/content/drive")
    BASE_DIR = "/content/drive/MyDrive/Libri_Vibevoice"
else:
    print("Detected: Local environment")
    # For local, try to detect the workspace path
    # This assumes the notebook is in the Libri_Vibevoice directory
    BASE_DIR = os.path.dirname(os.path.abspath("."))
    if "Libri_Vibevoice" not in BASE_DIR:
        # Fallback: use the known path structure
        BASE_DIR = r"G:\My Drive\Libri_Vibevoice"
    print(f"Using BASE_DIR: {BASE_DIR}")

# Set up paths
OUTPUTS_DIR = os.path.join(BASE_DIR, "outputs", "raw")
OUTPUTS_HUMANIZED_DIR = os.path.join(BASE_DIR, "outputs", "humanized_simple")

print(f"\nBASE_DIR: {BASE_DIR}")
print(f"OUTPUTS_DIR: {OUTPUTS_DIR}")
print(f"OUTPUTS_HUMANIZED_DIR: {OUTPUTS_HUMANIZED_DIR}")

# Verify outputs directory exists
if not os.path.isdir(OUTPUTS_DIR):
    raise FileNotFoundError(f"Outputs directory not found at: {OUTPUTS_DIR}")

# Create humanized outputs directory
os.makedirs(OUTPUTS_HUMANIZED_DIR, exist_ok=True)
print(f"\n✓ Setup complete. Outputs directory found: {OUTPUTS_DIR}")
print(f"✓ Humanized outputs will be saved to: {OUTPUTS_HUMANIZED_DIR}")



VOICE HUMANIZATION - Environment Setup
Detected: Local environment
Using BASE_DIR: G:\My Drive\Libri_Vibevoice

BASE_DIR: G:\My Drive\Libri_Vibevoice
OUTPUTS_DIR: G:\My Drive\Libri_Vibevoice\outputs
OUTPUTS_HUMANIZED_DIR: G:\My Drive\Libri_Vibevoice\outputs_humanized

✓ Setup complete. Outputs directory found: G:\My Drive\Libri_Vibevoice\outputs
✓ Humanized outputs will be saved to: G:\My Drive\Libri_Vibevoice\outputs_humanized


In [16]:
# Cell 2: Install Dependencies
# Install required packages for audio processing

import subprocess

print("=" * 80)
print("Installing Required Packages")
print("=" * 80)

packages = [
    "librosa>=0.10.0",
    "soundfile>=0.12.0",
    "scipy>=1.10.0",
    "numpy>=1.24.0",
    "tqdm>=4.65.0",
]

for package in packages:
    try:
        print(f"Installing {package}...")
        if IN_COLAB:
            subprocess.run([sys.executable, "-m", "pip", "install", package, "--quiet"], check=True)
        else:
            subprocess.run([sys.executable, "-m", "pip", "install", package], check=True)
        print(f"✓ {package} installed")
    except Exception as e:
        print(f"⚠ Warning: Failed to install {package}: {e}")

print("\n✓ Dependencies installation complete")



Installing Required Packages
Installing librosa>=0.10.0...
✓ librosa>=0.10.0 installed
Installing soundfile>=0.12.0...
✓ soundfile>=0.12.0 installed
Installing scipy>=1.10.0...
✓ scipy>=1.10.0 installed
Installing numpy>=1.24.0...
✓ numpy>=1.24.0 installed
Installing tqdm>=4.65.0...
✓ tqdm>=4.65.0 installed

✓ Dependencies installation complete


In [17]:
# Cell 3: Configuration
# Configure which output directories to process and humanization parameters

print("=" * 80)
print("Configuration")
print("=" * 80)

# ===== OUTPUT DIRECTORIES TO PROCESS =====
# Options:
# - List specific directories: ['25094444', '57b40817']
# - Process all: 'all'
OUTPUT_DIRECTORIES_TO_PROCESS = 'all'  # Change this to process specific directories

# ===== HUMANIZATION PARAMETERS =====
# Pause durations (in seconds)
PAUSE_AT_COMMA = 0.15      # 150ms pause at commas
PAUSE_AT_PERIOD = 0.4      # 400ms pause at periods
PAUSE_AT_QUESTION = 0.5    # 500ms pause at question marks
PAUSE_AT_EXCLAMATION = 0.5 # 500ms pause at exclamation marks

# Pitch variation (in semitones)
PITCH_VARIATION_MIN = -0.2  # Minimum pitch shift (reduced from -0.5 to prevent robotic sound)
PITCH_VARIATION_MAX = 0.2   # Maximum pitch shift (reduced from 0.5 to prevent robotic sound)

# Audio enhancement parameters
EQ_BOOST_FREQ_LOW = 2000    # Lower frequency for EQ boost (Hz)
EQ_BOOST_FREQ_HIGH = 4000   # Upper frequency for EQ boost (Hz)
EQ_BOOST_AMOUNT = 0.1       # Amount of EQ boost (reduced from 0.3 to prevent distortion)

# Compression parameters
COMPRESSION_RATIO = 2.0     # Compression ratio (reduced from 3.0 for gentler compression)
COMPRESSION_THRESHOLD = 0.85 # Threshold for compression (increased from 0.7 to compress less)

print(f"Output directories to process: {OUTPUT_DIRECTORIES_TO_PROCESS}")
print(f"Pause at comma: {PAUSE_AT_COMMA}s")
print(f"Pause at period: {PAUSE_AT_PERIOD}s")
print(f"Pitch variation range: {PITCH_VARIATION_MIN} to {PITCH_VARIATION_MAX} semitones")
print(f"EQ boost amount: {EQ_BOOST_AMOUNT} (reduced to prevent distortion)")
print(f"Compression ratio: {COMPRESSION_RATIO}:1 (gentler than before)")
print(f"Compression threshold: {COMPRESSION_THRESHOLD} (higher = less compression)")
print("\n✓ Configuration loaded with reduced processing to prevent distortion")



Configuration
Output directories to process: all
Pause at comma: 0.15s
Pause at period: 0.4s
Pitch variation range: -0.2 to 0.2 semitones
EQ boost amount: 0.1 (reduced to prevent distortion)
Compression ratio: 2.0:1 (gentler than before)
Compression threshold: 0.85 (higher = less compression)

✓ Configuration loaded with reduced processing to prevent distortion


In [18]:
# Cell 4: Humanization Function Definitions
# Core functions for humanizing audio

import librosa
import soundfile as sf
import numpy as np
from scipy import signal

def add_natural_pauses(audio, sr, text=None):
    """
    Add natural pauses to audio based on punctuation or regular intervals.
    
    Args:
        audio: Audio signal as numpy array
        sr: Sample rate
        text: Optional text string with punctuation for pause insertion
    
    Returns:
        Audio with pauses inserted
    """
    if text is None:
        # If no text provided, add subtle pauses at regular intervals
        # Split audio into chunks and add small pauses between them
        chunk_length = int(3.0 * sr)  # 3 second chunks
        chunks = []
        for i in range(0, len(audio), chunk_length):
            chunk = audio[i:i+chunk_length]
            chunks.append(chunk)
            # Add small pause between chunks (except after last chunk)
            if i + chunk_length < len(audio):
                pause_duration = int(0.1 * sr)  # 100ms pause
                pause = np.zeros(pause_duration)
                chunks.append(pause)
        return np.concatenate(chunks) if chunks else audio
    
    # If text is provided, insert pauses at punctuation
    # Note: This is a simplified version. For accurate pause placement,
    # you would need to align text with audio timing (requires forced alignment)
    # For now, we'll add pauses at regular intervals based on text length
    text_length = len(text)
    num_sentences = text.count('.') + text.count('!') + text.count('?')
    
    if num_sentences > 0:
        # Estimate pause positions based on audio length and sentence count
        audio_duration = len(audio) / sr
        pause_positions = np.linspace(0, audio_duration, num_sentences + 1)[1:-1]
        
        result = []
        last_pos = 0
        
        for pause_pos in pause_positions:
            pos_samples = int(pause_pos * sr)
            result.append(audio[last_pos:pos_samples])
            
            # Determine pause duration based on punctuation
            # This is simplified - in practice you'd need text-audio alignment
            pause_duration = int(PAUSE_AT_PERIOD * sr)
            pause = np.zeros(pause_duration)
            result.append(pause)
            last_pos = pos_samples
        
        result.append(audio[last_pos:])
        return np.concatenate(result) if result else audio
    
    return audio


def apply_pitch_variation(audio, sr):
    """
    Apply very subtle pitch variation to reduce monotony without sounding robotic.
    
    Args:
        audio: Audio signal as numpy array
        sr: Sample rate
    
    Returns:
        Audio with pitch variation applied
    """
    # Apply very subtle random pitch shift (reduced range)
    # Only apply if variation range is meaningful
    if abs(PITCH_VARIATION_MAX - PITCH_VARIATION_MIN) > 0.1:
        pitch_shift = np.random.uniform(PITCH_VARIATION_MIN, PITCH_VARIATION_MAX)
        try:
            audio_shifted = librosa.effects.pitch_shift(
                audio, 
                sr=sr, 
                n_steps=pitch_shift,
                bins_per_octave=12  # Standard tuning
            )
            return audio_shifted
        except Exception:
            # If pitch shift fails, return original
            return audio
    else:
        # If range is too small, skip pitch shifting to avoid artifacts
        return audio


def enhance_audio_quality(audio, sr):
    """
    Enhance audio quality with subtle EQ boost and gentle compression.
    Reduced processing to prevent distortion and robotic sound.
    
    Args:
        audio: Audio signal as numpy array
        sr: Sample rate
    
    Returns:
        Enhanced audio
    """
    # Normalize input first to prevent issues
    max_val_input = np.abs(audio).max()
    if max_val_input > 0:
        audio = audio / max_val_input * 0.9  # Normalize to 90% to leave headroom
    
    # Subtle EQ boost for clarity (2-4 kHz range) - only if amount > 0
    if EQ_BOOST_AMOUNT > 0:
        try:
            b, a = signal.butter(2, [EQ_BOOST_FREQ_LOW, EQ_BOOST_FREQ_HIGH], 
                                btype='band', fs=sr)
            audio_eq = signal.filtfilt(b, a, audio)
            
            # Blend original with EQ boosted signal (much more subtle now)
            audio = (1.0 - EQ_BOOST_AMOUNT) * audio + EQ_BOOST_AMOUNT * audio_eq
        except Exception:
            # If EQ fails, just use original audio
            pass
    
    # Gentle compression with smoother algorithm to reduce distortion
    threshold = COMPRESSION_THRESHOLD
    ratio = COMPRESSION_RATIO
    
    # Use smoother compression algorithm to reduce artifacts
    compressed_audio = audio.copy()
    
    # Only compress if there are significant peaks
    peak_mask = np.abs(audio) > threshold
    if np.any(peak_mask):
        # Calculate compression with smoother curve
        excess = np.abs(audio[peak_mask]) - threshold
        # Use softer knee for smoother compression
        compressed_amount = excess / ratio
        
        # Apply compression smoothly
        compressed_audio[peak_mask] = np.sign(audio[peak_mask]) * (
            threshold + compressed_amount
        )
    
    # Gentle normalization - preserve dynamics
    max_val = np.abs(compressed_audio).max()
    if max_val > 0:
        # Use gentler normalization to preserve natural dynamics
        compressed_audio = compressed_audio / max_val * 0.92  # Slightly less aggressive
    
    return compressed_audio


def humanize_audio(input_path, output_path, text=None):
    """
    Main function to humanize audio by applying all processing steps.
    
    Args:
        input_path: Path to input audio file
        output_path: Path to save humanized audio
        text: Optional text string for pause insertion
    
    Returns:
        True if successful, False otherwise
    """
    try:
        # Load audio
        audio, sr = librosa.load(input_path, sr=None)
        
        # Apply humanization steps
        # Step 1: Add natural pauses
        audio = add_natural_pauses(audio, sr, text)
        
        # Step 2: Apply pitch variation
        audio = apply_pitch_variation(audio, sr)
        
        # Step 3: Enhance audio quality
        audio = enhance_audio_quality(audio, sr)
        
        # Ensure output directory exists
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        
        # Save humanized audio
        sf.write(output_path, audio, sr)
        
        return True
    except Exception as e:
        print(f"  ✗ Error processing {input_path}: {e}")
        return False

print("✓ Humanization functions defined")



✓ Humanization functions defined


In [19]:
# Cell 5: Batch Processing
# Process all WAV files in the selected output directories

from pathlib import Path
from tqdm import tqdm

print("=" * 80)
print("Batch Processing - Humanizing Audio Files")
print("=" * 80)

# Determine which directories to process
if OUTPUT_DIRECTORIES_TO_PROCESS == 'all':
    # Get all subdirectories in outputs
    if os.path.isdir(OUTPUTS_DIR):
        directories_to_process = [d for d in os.listdir(OUTPUTS_DIR) 
                                if os.path.isdir(os.path.join(OUTPUTS_DIR, d))]
    else:
        directories_to_process = []
else:
    directories_to_process = OUTPUT_DIRECTORIES_TO_PROCESS

print(f"Found {len(directories_to_process)} directories to process: {directories_to_process}")

# Collect all WAV files to process
files_to_process = []

for run_id in directories_to_process:
    run_dir = os.path.join(OUTPUTS_DIR, run_id)
    if not os.path.isdir(run_dir):
        print(f"⚠ Warning: Directory not found: {run_dir}")
        continue
    
    # Find all WAV files in speaker subdirectories
    for speaker_id in os.listdir(run_dir):
        speaker_dir = os.path.join(run_dir, speaker_id)
        if not os.path.isdir(speaker_dir):
            continue
        
        for filename in os.listdir(speaker_dir):
            if filename.endswith('.wav'):
                input_path = os.path.join(speaker_dir, filename)
                # Create corresponding output path
                output_speaker_dir = os.path.join(OUTPUTS_HUMANIZED_DIR, run_id, speaker_id)
                output_path = os.path.join(output_speaker_dir, filename)
                files_to_process.append((input_path, output_path))

print(f"\nFound {len(files_to_process)} WAV files to process")

if len(files_to_process) == 0:
    print("⚠ No files found to process. Check your OUTPUT_DIRECTORIES_TO_PROCESS configuration.")
else:
    # Process files with progress bar
    successful = 0
    failed = 0
    
    for input_path, output_path in tqdm(files_to_process, desc="Humanizing audio"):
        if humanize_audio(input_path, output_path):
            successful += 1
        else:
            failed += 1
    
    print("\n" + "=" * 80)
    print("Processing Complete")
    print("=" * 80)
    print(f"✓ Successfully processed: {successful} files")
    if failed > 0:
        print(f"✗ Failed: {failed} files")
    print(f"\nHumanized files saved to: {OUTPUTS_HUMANIZED_DIR}")



Batch Processing - Humanizing Audio Files
Found 4 directories to process: ['25094444', '57b40817', '0fa80668', '3ac81b81']

Found 80 WAV files to process


Humanizing audio: 100%|██████████| 80/80 [00:34<00:00,  2.30it/s]


Processing Complete
✓ Successfully processed: 80 files

Humanized files saved to: G:\My Drive\Libri_Vibevoice\outputs_humanized


In [20]:
# Cell 6: Test Single File (Optional)
# Test humanization on a single file and play audio for comparison

print("=" * 80)
print("Test Single File")
print("=" * 80)

# Select a test file (modify these paths as needed)
TEST_RUN_ID = "57b40817"  # Change to test different run
TEST_SPEAKER_ID = "7794"   # Change to test different speaker
TEST_FILENAME = "simple_java_0.wav"  # Change to test different file

test_input_path = os.path.join(OUTPUTS_DIR, TEST_RUN_ID, TEST_SPEAKER_ID, TEST_FILENAME)
test_output_path = os.path.join(OUTPUTS_HUMANIZED_DIR, TEST_RUN_ID, TEST_SPEAKER_ID, 
                                 TEST_FILENAME.replace('.wav', '_test.wav'))

if not os.path.exists(test_input_path):
    print(f"⚠ Test file not found: {test_input_path}")
    print("Please update TEST_RUN_ID, TEST_SPEAKER_ID, or TEST_FILENAME above.")
else:
    print(f"Testing with: {test_input_path}")
    
    # Process the test file
    if humanize_audio(test_input_path, test_output_path):
        print(f"✓ Test file processed successfully")
        print(f"  Original: {test_input_path}")
        print(f"  Humanized: {test_output_path}")
        
        # Play audio if in Colab
        if IN_COLAB:
            from IPython.display import Audio, display
            print("\nPlaying original audio:")
            display(Audio(test_input_path))
            print("\nPlaying humanized audio:")
            display(Audio(test_output_path))
        else:
            print("\nTo listen to the audio, open the files:")
            print(f"  Original: {test_input_path}")
            print(f"  Humanized: {test_output_path}")
    else:
        print("✗ Failed to process test file")



Test Single File
Testing with: G:\My Drive\Libri_Vibevoice\outputs\57b40817\7794\simple_java_0.wav
✓ Test file processed successfully
  Original: G:\My Drive\Libri_Vibevoice\outputs\57b40817\7794\simple_java_0.wav
  Humanized: G:\My Drive\Libri_Vibevoice\outputs_humanized\57b40817\7794\simple_java_0_test.wav

To listen to the audio, open the files:
  Original: G:\My Drive\Libri_Vibevoice\outputs\57b40817\7794\simple_java_0.wav
  Humanized: G:\My Drive\Libri_Vibevoice\outputs_humanized\57b40817\7794\simple_java_0_test.wav


In [ ]:
# Cell 1: Environment Setup and Path Detection
# This cell detects whether you're running in Google Colab or locally
# and sets up the appropriate paths.

import os
import sys

# Detect environment
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("=" * 80)
print("VOICE HUMANIZATION - Environment Setup")
print("=" * 80)

if IN_COLAB:
    print("Detected: Google Colab environment")
    # Mount Google Drive
    print("\nMounting Google Drive...")
    drive.mount("/content/drive")
    BASE_DIR = "/content/drive/MyDrive/Libri_Vibevoice"
else:
    print("Detected: Local environment")
    # For local, try to detect the workspace path
    # This assumes the notebook is in the Libri_Vibevoice directory
    BASE_DIR = os.path.dirname(os.path.abspath("."))
    if "Libri_Vibevoice" not in BASE_DIR:
        # Fallback: use the known path structure
        BASE_DIR = r"G:\My Drive\Libri_Vibevoice"
    print(f"Using BASE_DIR: {BASE_DIR}")

# Set up paths
OUTPUTS_DIR = os.path.join(BASE_DIR, "outputs", "raw")
OUTPUTS_HUMANIZED_DIR = os.path.join(BASE_DIR, "outputs", "humanized_simple")

print(f"\nBASE_DIR: {BASE_DIR}")
print(f"OUTPUTS_DIR: {OUTPUTS_DIR}")
print(f"OUTPUTS_HUMANIZED_DIR: {OUTPUTS_HUMANIZED_DIR}")

# Verify outputs directory exists
if not os.path.isdir(OUTPUTS_DIR):
    raise FileNotFoundError(f"Outputs directory not found at: {OUTPUTS_DIR}")

# Create humanized outputs directory
os.makedirs(OUTPUTS_HUMANIZED_DIR, exist_ok=True)
print(f"\n✓ Setup complete. Outputs directory found: {OUTPUTS_DIR}")
print(f"✓ Humanized outputs will be saved to: {OUTPUTS_HUMANIZED_DIR}")



VOICE HUMANIZATION - Environment Setup
Detected: Local environment
Using BASE_DIR: G:\My Drive\Libri_Vibevoice

BASE_DIR: G:\My Drive\Libri_Vibevoice
OUTPUTS_DIR: G:\My Drive\Libri_Vibevoice\outputs
OUTPUTS_HUMANIZED_DIR: G:\My Drive\Libri_Vibevoice\outputs_humanized

✓ Setup complete. Outputs directory found: G:\My Drive\Libri_Vibevoice\outputs
✓ Humanized outputs will be saved to: G:\My Drive\Libri_Vibevoice\outputs_humanized
